In [ ]:
!nvidia-smi
!pip -q install -U "sentence-transformers>=2.7.0" "transformers>=4.51.0" accelerate pandas pyarrow tqdm

Tue Jun 16 15:16:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip -q uninstall -y pandas
!pip -q install "pandas==2.2.2"

In [ ]:
import pandas as pd
import torch

print("pandas:", pd.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

pandas: 2.2.2
CUDA: True
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path


BASE_DIR = Path("/content/drive/MyDrive/sme-legal-assistant")

CHUNKS_PATH = BASE_DIR / "processed/law_chunks.jsonl"
OUT_DIR = BASE_DIR / "data/embeddings/qwen3_06b"

OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
SHARD_SIZE = 5_000
BATCH_SIZE = 4
MAX_SEQ_LENGTH = 768
START_SHARD = 25
END_SHARD = 50

Mounted at /content/drive


In [ ]:
import json
import os
import uuid
import pandas as pd
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

QDRANT_POINT_NAMESPACE = uuid.UUID("734a5798-fdd0-406d-842d-2b3c446a8f28")

PAYLOAD_COLUMNS = [
    "chunk_id", "article_id", "subject_id", "subject_number", "subject_title",
    "topic_id", "topic_number", "topic_title", "chapter_title", "article_title",
    "clause_number", "point_label", "source_url", "chunk_type", "ordinal",
    "parent_chunk_id", "is_subchunk", "subchunk_index", "subchunk_count",
    "start_char", "end_char", "char_len", "word_count", "text",
]

def point_id(chunk_id: str) -> str:
    return str(uuid.uuid5(QDRANT_POINT_NAMESPACE, chunk_id))

def line(label, value):
    return f"{label}: {value}" if value else None

def build_embedding_text(row):
    parts = [
        line("Chủ đề", row.get("subject_title")),
        line("Đề mục", row.get("topic_title")),
        line("Chương", row.get("chapter_title")),
        line("Điều", row.get("article_title")),
        line("Khoản", row.get("clause_number")),
        line("Điểm", row.get("point_label")),
        line("Nguồn", row.get("source_url")),
        "",
        "Nội dung:",
        row.get("text") or "",
    ]
    return "\n".join(part for part in parts if part is not None).strip()

def iter_shards(path, shard_size):
    shard = []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            if not raw.strip():
                continue
            row = json.loads(raw)
            if not row.get("chunk_id") or not str(row.get("text") or "").strip():
                continue
            shard.append(row)
            if len(shard) >= shard_size:
                yield shard
                shard = []
    if shard:
        yield shard

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = MAX_SEQ_LENGTH

print("Device:", device)
print("Max seq length:", model.max_seq_length)

dim = model.get_sentence_embedding_dimension()
print("Embedding dim:", dim)
assert dim == 1024, f"Expected 1024, got {dim}"

for shard_idx, rows in enumerate(iter_shards(CHUNKS_PATH, SHARD_SIZE)):
    if shard_idx < START_SHARD:
        continue
    if END_SHARD is not None and shard_idx >= END_SHARD:
        break

    out_path = OUT_DIR / f"embedded_{shard_idx:03d}.parquet"

    if out_path.exists():
        print(f"Skip existing {out_path.name}")
        continue

    texts = [build_embedding_text(row) for row in rows]

    vectors = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype("float32")

    records = []
    for row, vector in zip(rows, vectors):
        record = {
            "point_id": point_id(str(row["chunk_id"])),
            "vector": vector.tolist(),
        }
        for col in PAYLOAD_COLUMNS:
            record[col] = row.get(col)
        records.append(record)

    df = pd.DataFrame(records)
    tmp_path = out_path.with_suffix(".tmp.parquet")
    df.to_parquet(tmp_path, index=False, engine="pyarrow")
    os.replace(tmp_path, out_path)

    print(f"Wrote {out_path.name}: {len(df):,} rows")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

Skip existing embedded_025.parquet
Skip existing embedded_026.parquet
Skip existing embedded_027.parquet
Skip existing embedded_028.parquet
Skip existing embedded_029.parquet
Skip existing embedded_030.parquet
Skip existing embedded_031.parquet
Skip existing embedded_032.parquet
Skip existing embedded_033.parquet
Skip existing embedded_034.parquet
Skip existing embedded_035.parquet
Skip existing embedded_036.parquet
Skip existing embedded_037.parquet
Skip existing embedded_038.parquet
Skip existing embedded_039.parquet
Skip existing embedded_040.parquet
Skip existing embedded_041.parquet
Skip existing embedded_042.parquet
Skip existing embedded_043.parquet
Skip existing embedded_044.parquet
Skip existing embedded_045.parquet
Skip existing embedded_046.parquet
Skip existing embedded_047.parquet


Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

Wrote embedded_048.parquet: 5,000 rows


Batches:   0%|          | 0/998 [00:00<?, ?it/s]

Wrote embedded_049.parquet: 3,991 rows


In [ ]:
import pyarrow.parquet as pq

files = sorted(OUT_DIR.glob("embedded_*.parquet"))
total_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in files)

expected_names = {f"embedded_{i:03d}.parquet" for i in range(50)}
actual_names = {f.name for f in files}

sample = pd.read_parquet(files[0], columns=["point_id", "vector"])

print("Số file:", len(files))
print("Tổng rows:", total_rows)
print("Thiếu file:", sorted(expected_names - actual_names))
print("Vector dimension:", len(sample.iloc[0]["vector"]))
print("File cuối rows:", pq.ParquetFile(files[-1]).metadata.num_rows)

Số file: 50
Tổng rows: 248991
Thiếu file: []
Vector dimension: 1024
File cuối rows: 3991


In [ ]:
files_done = len(sorted(OUT_DIR.glob("embedded_*.parquet")))
total_chunks = 248_991
total_files = (total_chunks + SHARD_SIZE - 1) // SHARD_SIZE

done_chunks = min(files_done * SHARD_SIZE, total_chunks)
remaining_chunks = total_chunks - done_chunks
remaining_batches = (remaining_chunks + BATCH_SIZE - 1) // BATCH_SIZE

print("Files done:", files_done)
print("Total files:", total_files)
print("Done chunks:", done_chunks)
print("Remaining chunks:", remaining_chunks)
print("Remaining batches:", remaining_batches)
print("Progress:", f"{done_chunks / total_chunks:.1%}")

Files done: 25
Total files: 50
Done chunks: 125000
Remaining chunks: 123991
Remaining batches: 30998
Progress: 50.2%


In [ ]:
files = sorted(OUT_DIR.glob("embedded_*.parquet"))
print("Số file đã xong:", len(files))

for f in files[:10]:
    print(f.name)

if files:
    print("File cuối:", files[-1].name)

Số file đã xong: 25
embedded_000.parquet
embedded_001.parquet
embedded_002.parquet
embedded_003.parquet
embedded_004.parquet
embedded_005.parquet
embedded_006.parquet
embedded_007.parquet
embedded_008.parquet
embedded_009.parquet
File cuối: embedded_024.parquet


In [ ]:
from pathlib import Path

files = sorted(OUT_DIR.glob("embedded_*.parquet")) + sorted(OUT_DIR.glob("embedded_*.tmp.parquet"))
print("Found:", len(files))

for f in files:
    print("Delete", f.name)
    f.unlink()

print("Done")

Found: 0
Done


In [ ]:
print("MODEL_NAME:", MODEL_NAME)
print("SHARD_SIZE:", SHARD_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)

print("CHUNKS_PATH exists:", CHUNKS_PATH.exists())
print("CHUNKS_PATH:", CHUNKS_PATH)
print("OUT_DIR:", OUT_DIR)
print("OUT_DIR exists:", OUT_DIR.exists())

MODEL_NAME: Qwen/Qwen3-Embedding-0.6B
SHARD_SIZE: 5000
BATCH_SIZE: 4
MAX_SEQ_LENGTH: 768
CHUNKS_PATH exists: True
CHUNKS_PATH: /content/drive/MyDrive/sme-legal-assistant/processed/law_chunks.jsonl
OUT_DIR: /content/drive/MyDrive/sme-legal-assistant/data/embeddings/qwen3_06b
OUT_DIR exists: True


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
!find "/content/drive/MyDrive" \( -iname "law_chunks*" -o -iname "*.jsonl" -o -iname "*.json" \) | head -100

/content/drive/MyDrive/sme-legal-assistant/processed/law_chunks.jsonl
